Import packages

In [47]:
import os
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import cobra
from cobra.io import read_sbml_model, write_sbml_model
from cobra.flux_analysis import flux_variability_analysis
from tqdm import tqdm

In [48]:
from pathlib import Path
import plotly
import plotly.express as px
import scipy.stats

os.environ["OMP_NUM_THREADS"] = "1"  # because of a data leak of KMeans on windows
import fba_comparison_stats as cmp
from efflux_method import *
import networkx as nx
import plotly.graph_objects as go
import plotly.io as pio

pio.renderers.default = "browser"

Import the models

How to make each models, how to put information in it? Efflux?

In [49]:
M_xanthus = read_sbml_model("../M_xanthus_model_V4.xml")

In [50]:
list_of_genes = []
for i in M_xanthus.genes:
    list_of_genes.append(i.id)

In [51]:
# Getting fluxes with constrains
dico = {
    14: "Alone",
    2: "P_E_coli_1-3",
    5: "P_B_subtilis",
    8: "P_Caulobacter",
    20: "P_E_coli_1-1",
}
for i in [14, 2, 5, 8, 20]:
    M_xanthus = read_sbml_model("../M_xanthus_model_V4.xml")
    DictAP = read_csv_data(
        "/home/mickael/github/M_xanthus-E_coli-Predation/data/Raw/WT_vs_4preys_iMAT.csv",
        id_gene=1,
        id_val=i,
        list_of_genes=list_of_genes,
        head=True,
        quantile=0.95,
    )
    Eflux(M_xanthus, DictAP, const=100, default_exp_val=1, ignore_human=True)
    M_xanthus.reactions.EX_glc_D_e.bounds = (0, 1000)
    model_c = M_xanthus.copy()
    write_sbml_model(
        model_c,
        "/home/mickael/github/M_xanthus-E_coli-Predation/fba_comparer-main/Models_Preda/M_xanthus_V4_efflux_"
        + dico[i]
        + ".xml",
    )
    print(dico[i] + " Done!")

KeyboardInterrupt: 

In [52]:
M_xanthus_alone = read_sbml_model("Models_Preda/M_xanthus_V4_efflux_Alone.xml")
solution_alone = M_xanthus_alone.optimize()

In [53]:
M_xanthus_predation = read_sbml_model(
    "Models_Preda/M_xanthus_V4_efflux_P_E_coli_1-3.xml"
)
solution_predation_E_3 = M_xanthus_predation.optimize()

In [54]:
M_xanthus_predation = read_sbml_model(
    "Models_Preda/M_xanthus_V4_efflux_P_B_subtilis.xml"
)
solution_predation_B = M_xanthus_predation.optimize()

In [55]:
M_xanthus_predation = read_sbml_model(
    "Models_Preda/M_xanthus_V4_efflux_P_Caulobacter.xml"
)
solution_predation_C = M_xanthus_predation.optimize()

In [56]:
M_xanthus_predation = read_sbml_model(
    "Models_Preda/M_xanthus_V4_efflux_P_E_coli_1-3.xml"
)
solution_predation_E_1 = M_xanthus_predation.optimize()

In [57]:
print(f"Alone:\t{solution_alone.objective_value}")
print(f"Predation E.coli 1/1 :\t{solution_predation_E_1.objective_value}")
print(f"Predation E.coli 1/3 :\t{solution_predation_E_3.objective_value}")
print(f"Predation B_subtilis:\t{solution_predation_B.objective_value}")
print(f"Predation Caulobacter:\t{solution_predation_C.objective_value}")

Alone:	0.204452758536134
Predation E.coli 1/1 :	0.10165517381424201
Predation E.coli 1/3 :	0.10165517381424201
Predation B_subtilis:	0.15158979104534828
Predation Caulobacter:	0.16752521568295686


## **FBA Comparer**
**Create combined dataframe and preprocess data**

Filter all reactions where fluxes are zero:

In [58]:
solutions = [
    solution_alone,
    solution_predation_E_1,
    solution_predation_E_3,
    solution_predation_B,
    solution_predation_C,
]
conditions = [
    "Alone",
    "Predation E.coli 1/1",
    "Predation E.coli 1/3",
    "Predation B. subtilis",
    "Predation Caulobacter",
]
obj_values = [
    solution_alone.objective_value,
    solution_predation_E_1.objective_value,
    solution_predation_E_3.objective_value,
    solution_predation_B.objective_value,
    solution_predation_C.objective_value,
]

mxanthus = cmp.build_dataframe(M_xanthus_alone, solutions, conditions)
mxanthus_filtered = cmp.filter_dataframe(mxanthus, conditions, rounding=True)
print(f"Number of reactions in the models: {len(mxanthus)}")
print(f"Number of nonzero reactions in the models: {len(mxanthus_filtered)}")

Number of reactions in the models: 1339
Number of nonzero reactions in the models: 445


normalize data:

In [59]:
mxanthus_normalized = cmp.normalize_dataframe_cols(
    mxanthus_filtered, conditions, obj_values
)  # biomass normalization
mxanthus_normalized_div_by_max = cmp.normalize_dataframe_rows(
    mxanthus_normalized, conditions, "div_by_max"
)  # normalization reactions
# note: there is no difference between the order of div_by_max normalization and filtering for changing reactions
mxanthus_normalized_div_by_max_changing = mxanthus_normalized_div_by_max[
    mxanthus_normalized_div_by_max["Std_dev"] >= 0.01
]  # keep only reactions that change
print(
    f"Number of normalized changing reactions: {len(mxanthus_normalized_div_by_max_changing)}"
)

Number of normalized changing reactions: 194


## Identify reactions with most/least variation

Most affected / change reaction

In [60]:
most_variable_bar_plot = cmp.bar_plot_flux_variation(
    mxanthus_normalized, conditions, 0, 20
)
most_variable_bar_plot.show()

In [61]:
sorted_flux = mxanthus_normalized[["ID", "Name", "Std_dev"]].sort_values(
    by=["Std_dev"], ascending=False
)
sorted_flux["Kegg"] = [
    M_xanthus_alone.reactions.get_by_id(id).__dict__["_annotation"].get("kegg.reaction")
    for id in sorted_flux["ID"]
]
sorted_flux[0:20]

,ID,Name,Std_dev,Kegg
842,rxn01507_c,2'-Deoxyadenosine 5'-monophosphate phosphohydr...,5.197087,R02088
583,rxn01508_c,ATP:deoxyadenosine 5'-phosphotransferase [c],5.197087,R02089
716,rxn01366_c,Uridine:phosphate alpha-D-ribosyltransferase [c],3.538081,R01876
910,rxn00709_c,ATP:uridine 5'-phosphotransferase [c],3.538081,R00964
1067,rxn00117_c,ATP:UDP phosphotransferase [c],3.365245,R00156
352,rxn00463_c,Uridine triphosphate pyrophosphohydrolase [c],3.365245,R00662
909,rxn00119_c,ATP:UMP phosphotransferase [c],3.281387,R00158
499,rxn00711_c,UMP:diphosphate phospho-alpha-D-ribosyltransfe...,3.103785,R00966
1261,EX_pi_e,Exchange for Phosphate [e],3.093527,None
672,rxn05312_c,Inorganic phosphate transporter [c],3.093527,None


Pathway analysis

In [62]:
from bioservices import KEGG

k = KEGG()

for rxn in M_xanthus.reactions:
    if "kegg.reaction" in rxn.annotation:
        kegg_id = rxn.annotation["kegg.reaction"]
        try:
            data = k.get(kegg_id)
            parsed = k.parse(data)
            if "PATHWAY" in parsed:
                rxn.subsystem = list(parsed["PATHWAY"].values())[0]
        except:
            continue

WARNING [bioservices.KEGG:535]:  HTTP 404 Not Found (https://rest.kegg.jp/get/R02371)
WARNING [bioservices.KEGG:1210]:  Could not parse the entry correctly.
WARNING [bioservices.KEGG:535]:  HTTP 404 Not Found (https://rest.kegg.jp/get/R01549)
WARNING [bioservices.KEGG:1210]:  Could not parse the entry correctly.
WARNING [bioservices.KEGG:535]:  HTTP 404 Not Found (https://rest.kegg.jp/get/R02097)
WARNING [bioservices.KEGG:1210]:  Could not parse the entry correctly.
WARNING [bioservices.KEGG:535]:  HTTP 404 Not Found (https://rest.kegg.jp/get/R02292)
WARNING [bioservices.KEGG:1210]:  Could not parse the entry correctly.
WARNING [bioservices.KEGG:535]:  HTTP 404 Not Found (https://rest.kegg.jp/get/R00962)
WARNING [bioservices.KEGG:1210]:  Could not parse the entry correctly.
WARNING [bioservices.KEGG:535]:  HTTP 404 Not Found (https://rest.kegg.jp/get/R00967)
WARNING [bioservices.KEGG:1210]:  Could not parse the entry correctly.
WARNING [bioservices.KEGG:535]:  HTTP 404 Not Found (https

In [63]:
pathway_dict_M = {}
for i in M_xanthus.reactions:
    if i.subsystem in pathway_dict_M:
        pathway_dict_M[i.subsystem].append(i.id)
    else:
        pathway_dict_M[i.subsystem] = [i.id]

In [64]:
pw_list = cmp.pathway_variability(
    mxanthus_normalized, pathway_dict_M, ascending=True, num=20
)

Pyrimidine metabolism		 -> 0.92 (contains 25 reactions)
Purine metabolism		 -> 0.90 (contains 35 reactions)
Arginine biosynthesis		 -> 0.57 (contains 7 reactions)
Glycerophospholipid metabolism		 -> 0.54 (contains 1 reactions)
Glycolysis / Gluconeogenesis		 -> 0.49 (contains 6 reactions)
Metabolic pathways		 -> 0.35 (contains 1 reactions)
Citrate cycle (TCA cycle)		 -> 0.34 (contains 8 reactions)
Alanine, aspartate and glutamate metabolism		 -> 0.25 (contains 4 reactions)
Glyoxylate and dicarboxylate metabolism		 -> 0.17 (contains 3 reactions)
		 -> 0.16 (contains 152 reactions)
Pyruvate metabolism		 -> 0.14 (contains 1 reactions)
Phenylalanine metabolism		 -> 0.13 (contains 2 reactions)
Fatty acid elongation		 -> 0.10 (contains 6 reactions)
Fatty acid degradation		 -> 0.10 (contains 1 reactions)
Glycine, serine and threonine metabolism		 -> 0.09 (contains 9 reactions)
Cysteine and methionine metabolism		 -> 0.07 (contains 5 reactions)
One carbon pool by folate		 -> 0.07 (contains 5 re

In [65]:
for i in M_xanthus.reactions._dict:
    if M_xanthus.reactions.get_by_id(i).subsystem == "Fatty acid degradation":
        print(i)

rxn02679_c
rxn01451_c
rxn02720_c
rxn00178_c
rxn03253_c
rxn01802_c
rxn00872_c
rxn02803_c
rxn02345_c
rxn00946_c
rxn03251_c
rxn02167_c


## Flux correlation network

In [66]:
corr_matrix = cmp.flux_coupling_matrix(
    mxanthus_normalized_div_by_max, conditions, remove_unchanging=0.01, abs_values=True
)

251 reaction were removed, because their Std_dev is lower than 0.01


In [ ]:
# Correlation network
nw = cmp.flux_coupling_network(corr_matrix, pc=0.8)
fig = cmp.visualize_interactive_network(nw, mxanthus_normalized_div_by_max, conditions)
fig.show()

Created Flux Coupling network with pc threshold 0.8.
Number of nodes: 194
Number of edges: 3419
Number of connected components: 3


In [ ]:
# Colored pathway network
nw3 = nw
cmp.color_network_by_pathway(
    nw3, pathway= "Pyrimidine metabolism", pathway_dict= pathway_dict_M
)  # takes edge weights into account!

fig = cmp.visualize_interactive_network(nw3, mxanthus_normalized_div_by_max, conditions)
fig.show()

In [ ]:
# Colored community network
nw2 = nw
cmp.color_network_by_communities(
    nw2, 6, include_weights=True
)  # takes edge weights into account!

comm_map = cmp.get_community_list(nw)
color = cmp.get_color_map(comm_map)

fig = cmp.visualize_interactive_network(nw2, mxanthus_normalized_div_by_max, conditions)
fig.show()

Community map

In [86]:
fig = cmp.community_bar_plots(mxanthus_normalized_div_by_max, conditions, comm_map)
fig.show()

In [87]:
fig = cmp.community_box_plots(mxanthus_normalized_div_by_max, conditions, comm_map)
fig.show()